### **Enviornment**

In [ ]:
# Setting up the local imports and env variables
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGCHAIN_TRACING_V2'] = os.getenv("LANGCHAIN_TRACING_V2")
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

# **Logical and Semantic routing**

### **Logical Routing**

In [1]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_ollama import OllamaEmbeddings , ChatOllama

class RouteQuery(BaseModel):
    """ Route a user query to the most relevant data source. """
    data_source : Literal["python_docs", "js_docs", "golang_docs"] = Field(... , description="Given a user question choose which datasource would be most relevant for answering their question")

# LLM with function call 
llm = ChatOllama(model="llama3.2:3b", temperature=0.2)
structured_llm = llm.with_structured_output(RouteQuery)

# Prompt 
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ('system' , system),
        ('human' , '{question}')
    ]
)

# Define Prompt 
router = prompt | structured_llm

In [2]:
question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})

In [3]:
result

RouteQuery(data_source='python_docs')

In [4]:
result.data_source

'python_docs'

In [9]:
def choose_route(result):
    if "python_docs" in result.data_source.lower():
        return "chain for python_docs"
    elif "js_docs" in result.data_source.lower():
        return "chain for js_docs"
    else:
        return "golang_docs"
    
from langchain_core.runnables import RunnableLambda

full_chain = router | RunnableLambda(choose_route)

In [10]:
full_chain.invoke({"question": question})

'chain for python_docs'

In [11]:
sample_chain = router | choose_route

### **Semantic Routing**

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.prompts import ChatPromptTemplate , PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import OllamaEmbeddings , ChatOllama

# Two prompts
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

prompt_templates = [physics_template , math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

# Route Question to Prompt 
def prompt_router(input):
    # Embed Question
    query_embedding = embeddings.embed_query(input['query'])
    # Compute similarity
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return ChatPromptTemplate.from_messages([
        ("human", most_similar)
    ])


chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

print(chain.invoke("What's a black hole"))

Using PHYSICS
Black holes! One of the most fascinating and mysterious objects in the universe.

A black hole is essentially a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself and its gravity becomes so strong that it warps the fabric of spacetime around it.

Imagine spacetime as a trampoline. If you place a heavy object, like a bowling ball, on the trampoline, it will warp and curve, creating a depression. Now imagine taking away all the air from the trampoline, making it even more rigid and dense. That's roughly what happens when a star collapses into a black hole.

The point of no return, called the event horizon, marks the boundary beyond which anything that enters cannot escape. Once inside, you're trapped by the black hole's gravity, and your journey to the center would be... well, let's just say it wouldn't be pleasant!

Black holes come in different sizes, ranging from smal

# **Query Construction**

### **Query structuring for metadata filters**

In [28]:
from langchain_community.document_loaders import YoutubeLoader
from yt_dlp import YoutubeDL
from datetime import datetime

url = "https://www.youtube.com/watch?v=pbAd8O1Lvm4"

# Load transcript only
docs = YoutubeLoader.from_youtube_url(
    url,
    add_video_info=False
).load()

# Fetch metadata using yt-dlp
with YoutubeDL({"quiet": True}) as ydl:
    info = ydl.extract_info(url, download=False)

docs[0].metadata.update({
    "source": info["id"],
    "title": info["title"],
    "description": info.get("description", "Unknown"),
    "view_count": info.get("view_count", 0),
    "thumbnail_url": info.get("thumbnail", "Unknown"),
    "publish_date": (
        datetime.strptime(info["upload_date"], "%Y%m%d").strftime("%Y-%m-%d %H:%M:%S")
        if info.get("upload_date")
        else "Unknown"
    ),
    "length": info.get("duration", 0),
    "author": info.get("uploader", "Unknown"),
})

print(docs[0].metadata)

{'source': 'pbAd8O1Lvm4', 'title': 'Self-reflective RAG with LangGraph: Self-RAG and CRAG', 'description': 'Self-reflection can greatly enhance RAG, enabling correction of poor quality retrieval or generations. Several recent RAG papers focus on this theme, but implementing the ideas can be tricky. Here, we show that LangGraph can be easily used for "flow engineering" of self-reflective RAG pipelines. We provide cookbooks for implementing ideas from two interesting papers, Self-RAG and C-RAG.\n\nCode:\nhttps://github.com/langchain-ai/langgraph/tree/main/examples/rag', 'view_count': 38569, 'thumbnail_url': 'https://i.ytimg.com/vi/pbAd8O1Lvm4/maxresdefault.jpg', 'publish_date': '2024-02-07 00:00:00', 'length': 1058, 'author': 'LangChain'}


In [29]:
import datetime 
from typing import Literal, Optional , Tuple 
from pydantic import BaseModel , Field 

class TutorialSearch(BaseModel):
    """Search over a database of tutorial videos about a software library."""

    content_search: str = Field(
        ...,
        description="Similarity search query applied to video transcripts.",
    )
    title_search: str = Field(
        ...,
        description=(
            "Alternate version of the content search query to apply to video titles. "
            "Should be succinct and only include key words that could be in a video "
            "title."
        ),
    )
    min_view_count: Optional[int] = Field(
        None,
        description="Minimum view count filter, inclusive. Only use if explicitly specified.",
    )
    max_view_count: Optional[int] = Field(
        None,
        description="Maximum view count filter, exclusive. Only use if explicitly specified.",
    )
    earliest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Earliest publish date filter, inclusive. Only use if explicitly specified.",
    )
    latest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Latest publish date filter, exclusive. Only use if explicitly specified.",
    )
    min_length_sec: Optional[int] = Field(
        None,
        description="Minimum video length in seconds, inclusive. Only use if explicitly specified.",
    )
    max_length_sec: Optional[int] = Field(
        None,
        description="Maximum video length in seconds, exclusive. Only use if explicitly specified.",
    )

    def pretty_print(self) -> None:
        for field in self.__fields__:
            if getattr(self, field) is not None and getattr(self, field) != getattr(
                self.__fields__[field], "default", None
            ):
                print(f"{field}: {getattr(self, field)}")

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama,OllamaEmbeddings

system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

llm = ChatOllama(model="llama3.2:3b", temperature=0.2)
structured_llm = llm.with_structured_output(TutorialSearch)
query_analyzer = prompt | structured_llm

In [31]:
query_analyzer.invoke({"question": "rag from scratch"}).pretty_print()

content_search: SELECT * FROM tutorials WHERE title LIKE '%Rag% From Scratch%' OR description LIKE '%Rag% From Scratch%'
title_search: SELECT * FROM tutorials WHERE title LIKE '%Rag% From Scratch%'
latest_publish_date: 2022-01-01


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13908\3985746125.py:46: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for field in self.__fields__:
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13908\3985746125.py:48: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  self.__fields__[field], "default", None


In [32]:
query_analyzer.invoke(
    {"question": "videos on chat langchain published in 2023"}
).pretty_print()

content_search: langchain AND 'Chat' AND (published:2023)
title_search: LangChain Chat
max_length_sec: 60


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13908\3985746125.py:46: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for field in self.__fields__:
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13908\3985746125.py:48: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  self.__fields__[field], "default", None


In [33]:
query_analyzer.invoke(
    {"question": "videos that are focused on the topic of chat langchain that are published before 2024"}
).pretty_print()

content_search: topic:chat langchain AND date_published:<2024-01-01
title_search: topic:chat langchain AND title:*
max_length_sec: 300


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13908\3985746125.py:46: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for field in self.__fields__:
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13908\3985746125.py:48: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  self.__fields__[field], "default", None


In [34]:
query_analyzer.invoke(
    {
        "question": "how to use multi-modal models in an agent, only videos under 5 minutes"
    }
).pretty_print()

content_search: multi-modal models agent
title_search: using multi-modal models in agent
min_length_sec: 300
max_length_sec: 300


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13908\3985746125.py:46: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for field in self.__fields__:
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13908\3985746125.py:48: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  self.__fields__[field], "default", None
